In [32]:
import re
import string
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Descargas necesarias
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# Para mostrar tablas
from IPython.display import display

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gianp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gianp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gianp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [33]:
# ==========================
# Carga del corpus
# ==========================

df = pd.read_csv("df_total.csv")

print("Forma del dataset:")
print(df.shape)

display(df.head())

Forma del dataset:
(1217, 3)


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra


In [34]:
print("Número de documentos:", len(df))

print("\nColumnas del dataset:")
print(df.columns.tolist())

print("\nValores nulos:")
display(df.isnull().sum())

print("\nDuplicados completos:", df.duplicated().sum())

print("Duplicados únicamente en la noticia:")
print(df["news"].duplicated().sum())

print("\nNúmero de categorías:")
print(df["Type"].nunique())

display(df["Type"].value_counts())

Número de documentos: 1217

Columnas del dataset:
['url', 'news', 'Type']

Valores nulos:


url     0
news    0
Type    0
dtype: int64


Duplicados completos: 75
Duplicados únicamente en la noticia:
79

Número de categorías:
7


Type
Macroeconomia     340
Alianzas          247
Innovacion        195
Regulaciones      142
Sostenibilidad    137
Otra              130
Reputacion         26
Name: count, dtype: int64

In [35]:
# ==========================
# Limpieza inicial
# ==========================

# Eliminar espacios
df["news"] = df["news"].astype(str).str.strip()
df["Type"] = df["Type"].astype(str).str.strip()

# Eliminar noticias vacías
df = df[df["news"] != ""].copy()

# Eliminar noticias duplicadas
df = df.drop_duplicates(subset=["news"]).reset_index(drop=True)

print("Cantidad de documentos después de limpiar:", len(df))

Cantidad de documentos después de limpiar: 1137


In [36]:
SPANISH_STOPWORDS = set(stopwords.words("spanish"))

PUNCT = set(string.punctuation)
PUNCT.update({"¿","¡","«","»","“","”","‘","’","…","—","–"})


def tokenize(text):
    return word_tokenize(text, language="spanish")


def lowercase(tokens):
    return [t.lower() for t in tokens]


def remove_punctuation(tokens):
    clean = []

    for token in tokens:

        if token in PUNCT:
            continue

        if re.fullmatch(r"[\W_]+", token):
            continue

        clean.append(token)

    return clean


def remove_stopwords(tokens):

    return [t for t in tokens if t not in SPANISH_STOPWORDS]

In [37]:
# ====================================
# Lematización usando spaCy
# ====================================

try:
    import spacy

    try:
        nlp = spacy.load("es_core_news_sm")

    except:

        print("No está instalado el modelo de español.")
        print("Ejecuta:")
        print("python -m spacy download es_core_news_sm")

        nlp = None

except:

    print("spaCy no está instalado.")
    nlp = None


def lemmatize(tokens):

    if nlp is None:
        return tokens

    doc = nlp(" ".join(tokens))

    return [tok.lemma_.lower() for tok in doc if tok.lemma_.strip() != ""]

In [38]:
def flatten(list_of_lists):

    return [word for doc in list_of_lists for word in doc]


def corpus_stats(documents):

    words = flatten(documents)

    return {

        "tokens": len(words),

        "types": len(set(words))

    }

In [39]:
# Tokenización
df["tokens_1"] = df["news"].apply(tokenize)

# Minúsculas
df["tokens_2"] = df["tokens_1"].apply(lowercase)

# Eliminar puntuación
df["tokens_3"] = df["tokens_2"].apply(remove_punctuation)

# Eliminar stopwords
df["tokens_4"] = df["tokens_3"].apply(remove_stopwords)

# Lematización
df["tokens_5"] = df["tokens_4"].apply(lemmatize)

In [40]:
pipeline = {

    "Tokenización": df["tokens_1"],

    "Minúsculas": df["tokens_2"],

    "Sin puntuación": df["tokens_3"],

    "Sin stopwords": df["tokens_4"],

    "Lematización": df["tokens_5"]

}

resultado = []

for etapa, docs in pipeline.items():

    stats = corpus_stats(docs.tolist())

    resultado.append({

        "Etapa": etapa,

        "Tokens": stats["tokens"],

        "Tipos": stats["types"]

    })

pipeline_df = pd.DataFrame(resultado)

display(pipeline_df)

,Etapa,Tokens,Tipos
0,Tokenización,631733,38947
1,Minúsculas,631733,36099
2,Sin puntuación,598368,36063
3,Sin stopwords,318944,35852
4,Lematización,319410,28804
